# Managing Resources in Azure ML

## Notebook Setup

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
your_project_path = "/home/azureuser/cloudfiles/code/Users/dominik.mika/dp100-learn/"
os.chdir(your_project_path)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Found the config file in: /config.json


## Manage Data

### Datastores

In [8]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

dmdp100
workspaceworkingdirectory
workspaceblobstore
workspaceartifactstore
workspacefilestore


#### Create a datastore

You can create a new datastore in a default azure ml storage account container. However you can also create a new container in an existing storage account and register it as a datastore. You can do it in azure portal or programmatically as shown below.

##### Create a data container in existing storage account

In [4]:
from azure.storage.blob import BlobServiceClient
from utils.consts import AZUREML_STORAGE_ACCOUNT_NAME, AZUREML_STORAGE_ACCOUNT_ACCESS_KEY

storage_container_name = "dmdp100-azureml"

# Build connection string
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={AZUREML_STORAGE_ACCOUNT_NAME};"
    f"AccountKey={AZUREML_STORAGE_ACCOUNT_ACCESS_KEY};EndpointSuffix=core.windows.net"
)

# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create container (if not exists)
try:
    blob_service_client.create_container(storage_container_name)
    print(f"Container '{storage_container_name}' created.")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{storage_container_name}' already exists.")
    else:
        raise

Container 'dmdp100-azureml' created.


##### Create a datastore

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from utils.consts import AZUREML_STORAGE_ACCOUNT_NAME, AZUREML_STORAGE_ACCOUNT_ACCESS_KEY

datastore_name = "dmdp100"
storage_container_name = "dmdp100-azureml"

store = AzureBlobDatastore(
    name=datastore_name, # name cannot contain "-"
    description="Blob Storage for DP-100 certification prep",
    account_name=AZUREML_STORAGE_ACCOUNT_NAME,
    container_name=storage_container_name, 
    credentials=AccountKeyConfiguration(
        account_key=AZUREML_STORAGE_ACCOUNT_ACCESS_KEY
    ),
    type="azure_blob",
)

ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'dmdp100', 'description': 'Blob Storage for DP-100 certification prep', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/datastores/dmdp100', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7a5cf8b82500>, 'credentials': {'type': 'account_key'}, 'container_name': 'dmdp100-azureml', 'account_name': 'polandaidevmlst', 'endpoint': 'core.windows.net', 'protocol': 'https'})

##### (Optional) Set a datastore as default


In [27]:
print(f"Current default datastore: {ml_client.datastores.get_default().name}")

Current default datastore: workspaceblobstore


In [ ]:
# Doesn't work
# from utils.consts import AZUREML_WORKSPACE_NAME
# datastore_name = "dmdp100"
# workspace = ml_client.workspaces.get(AZUREML_WORKSPACE_NAME)
# workspace.default_datastore = store
# ml_client.workspaces.begin_update(workspace).result()
# print(f"Default datastore: {ml_client.datastores.get_default().name}")

Workspace({'kind': 'default', 'print_as_yaml': False, 'discovery_url': 'https://westeurope.api.azureml.ms/discovery', 'mlflow_tracking_uri': 'azureml://westeurope.api.azureml.ms/mlflow/v1.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw', 'workspace_id': '98a4e2ed-9e4a-44b6-a65e-e3273c9dc09b', 'feature_store_settings': None, 'name': 'polandaidevml-mlw', 'description': '', 'tags': {}, 'properties': {}, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7a5ce0a45d50>, 'display_name': '', 'location': 'westeurope', 'resource_group': 'polan

### Data Assets

#### Creating Data Assets

In [30]:
datastore_name = "dmdp100"

##### URI_FILE

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/azure-ml-labs-data/diabetes/diabetes.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="diabetes-data-file",
)

ml_client.data.create_or_update(my_data)

Uploading diabetes.csv (< 1 MB): 100%|██████████| 518k/518k [00:00<00:00, 17.6MB/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/3d8efe6c7dafbd432031e3f030cc92dd5acd9da03214f6d3ac06b02ed81cc551/diabetes.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-file', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/diabetes-data-file/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <az

In [31]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/telco-churn-data/telco-customer-churn.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="telco-churn-file-raw"
)

ml_client.data.create_or_update(my_data)

Uploading telco-customer-churn.csv (< 1 MB): 100%|██████████| 970k/970k [00:00<00:00, 28.8MB/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/c9f74bf3dd2417bba280f0ceef276024cbe95a3935ed06a94a7f357d15495775/telco-customer-churn.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-file-raw', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/telco-churn-file-raw/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creati

##### URI_FOLDER

In [32]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = './data/telco-churn-data'

my_data = Data(
    path=folder_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Data asset pointing to data-asset-path folder in datastore",
    name="telco-churn-folder-raw",
)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.97 MBs): 100%|██████████| 970588/970588 [00:00<00:00, 19806944.60it/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/6b41ae31c618bc8c542df493b94c95d2c0ad3f6843ff934bcd1b6294be86382e/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-folder-raw', 'description': 'Data asset pointing to data-asset-path folder in datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/telco-churn-folder-raw/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <azure.ai.ml.e

##### MLTABLE

In [33]:
%%writefile data/azure-ml-labs-data/diabetes/MLTable

paths:
  - file: ./diabetes.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/azure-ml-labs-data/diabetes/MLTable


In [35]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/azure-ml-labs-data/diabetes'

my_data = Data(
    path=data_path,
    datastore=datastore_name,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to diabetes.csv in data folder",
    name="diabetes-data-table",
)

ml_client.data.create_or_update(my_data)

Uploading diabetes (0.52 MBs): 100%|██████████| 517871/517871 [00:00<00:00, 8885768.78it/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/60aba111c79a4a33c719346aff233bf95950caa2dccbc2f200fe25c59119fad6/diabetes/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./diabetes.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-table', 'description': 'MLTable pointing to diabetes.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/diabetes-data-table/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <azure.ai.ml.entities._syste

In [36]:
%%writefile data/telco-churn-data/MLTable

paths:
  - file: ./telco-customer-churn.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/telco-churn-data/MLTable


In [37]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/telco-churn-data'

my_data = Data(
    path=data_path,
    datastore=datastore_name,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to telco-customer-churn.csv in data folder",
    name="telco-churn-table-raw",

)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/6b41ae31c618bc8c542df493b94c95d2c0ad3f6843ff934bcd1b6294be86382e/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./telco-customer-churn.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-table-raw', 'description': 'MLTable pointing to telco-customer-churn.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/telco-churn-table-raw/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_co

#### Read the Data Assets Locally

In [2]:
import pandas as pd
import mltable

In [3]:
data_asset = ml_client.data.get("telco-churn-file-raw", version="1")
df = pd.read_csv(data_asset.path)
df.sample(2)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
1705,4918-FYJNT,Female,1,Yes,No,55,Yes,Yes,Fiber optic,No,...,Yes,Yes,No,No,Month-to-month,No,Electronic check,90.45,5044.8,No
1241,0096-FCPUF,Male,0,No,No,30,Yes,Yes,DSL,Yes,...,No,No,No,Yes,Month-to-month,Yes,Mailed check,64.50,1888.45,No


In [4]:
data_asset = ml_client.data.get("telco-churn-folder-raw", version="1")
path = {
  'folder': data_asset.path
}
# tbl = mltable.from_delimited_files(paths=[path])
# df = tbl.to_pandas_dataframe()
df = pd.read_csv(os.path.join(data_asset.path, 'telco-customer-churn.csv'))
df.sample(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
890,5898-IGSLP,Male,0,Yes,Yes,31,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,No,No,Month-to-month,No,Electronic check,89.3,2823,No
3380,5178-LMXOP,Male,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.1,95.1,Yes


In [5]:
data_asset = ml_client.data.get("diabetes-data-table", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,False
1,1147438,8,92,93,47,36,21.240576,0.158365,23,False
2,1640031,7,115,47,52,35,41.511523,0.079019,23,False
3,1883350,9,103,78,25,304,29.582192,1.282870,43,True
4,1424119,1,85,59,27,35,42.604536,0.549542,22,False


In [6]:
data_asset = ml_client.data.get("telco-churn-table-raw", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,...,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,...,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,...,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,False
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,...,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,True


## Manage Compute and Environments

##